Hay que generar una mascara dinámica para cada buque de la posición mayoritaria mensual.

In [9]:
import pandas as pd
import numpy as np
import xarray as xr
import json

In [10]:
with open("../../data/bbox.json", "r") as f:
    bbox = json.load(f)
min_lon, min_lat, max_lon, max_lat = [bbox["min_lon"], bbox["min_lat"], bbox["max_lon"], bbox["max_lat"]]

res = 0.125

filepath = "../../../../ARVI-IEO/IEO/DATA_ANALYSIS/unir_db_py/data_output/2005-2025.csv"
df = pd.read_csv(filepath, low_memory=False)
df["FechaCaptura"] = pd.to_datetime(df["FechaCaptura"], format="mixed")
df= df[df['FechaCaptura'].between('2025-01-01', '2025-12-31')]

mask_ds = xr.open_dataset(f"../../data/processed/static/area_pesca_{res}.nc")

real_ds = xr.open_dataset(f"../../data/processed/targets/cpue_{res}.nc")
real_ds = real_ds.sel(time=slice("2025-01-01", "2025-12-31"))
df.columns

Index(['IdCaptura', 'NombreBuque', 'CodigoBuque', 'ZonaFAO', 'ArteFAO',
       'FechaCaptura', 'Lat', 'Lon', 'NumOperaciones', 'TiempoPesca',
       'EspecieFAO', 'ValorMalla', 'Peso'],
      dtype='object')

In [11]:
df1 = df[["FechaCaptura","NombreBuque", "Lat", "Lon", "Peso", "TiempoPesca", "NumOperaciones", "EspecieFAO" ]] # Select only the necessary columns
df1= df1.dropna(subset=["Lat", "Lon", "TiempoPesca"])
df1[["Lat", "Lon"]] = df1[["Lat", "Lon"]].apply(lambda x: -abs(x)) #correccion de las posiciones

df1 = df1[
    (df1['Lon'] >= min_lon) & (df1['Lon'] <= max_lon) &
    (df1['Lat'] >= min_lat) & (df1['Lat'] <= max_lat)
    ] #solo datos del area de interés 
df1 = df1.dropna(subset=["Peso", "TiempoPesca"])

lances = df1[["FechaCaptura", "NombreBuque", "Lat", "Lon", "TiempoPesca", "NumOperaciones"]].drop_duplicates()
lances['Fecha'] = lances['FechaCaptura'].dt.to_period('M').dt.to_timestamp()
effort = lances.groupby(["NombreBuque", "Fecha", "Lat", "Lon"]).agg(Horas=('TiempoPesca', 'sum')).reset_index()
effort["Horas"] = effort["Horas"].div(60)

In [12]:

lat_grid = np.arange(min_lat, max_lat + res, res)
lon_grid = np.arange(min_lon, max_lon + res, res)

effort["lat_bin"] = pd.cut(effort["Lat"], bins=lat_grid, labels=lat_grid[:-1])
effort["lon_bin"] = pd.cut(effort["Lon"], bins=lon_grid, labels=lon_grid[:-1])

effort_grid = effort.groupby(["NombreBuque",'Fecha', "lat_bin", "lon_bin"], observed=True)['Horas'].sum().reset_index() 

dsxr_effort = (
    effort_grid
    .set_index(["NombreBuque","Fecha", "lat_bin", "lon_bin"])
    .to_xarray()
    .rename({"Fecha": "time", "lat_bin":"lat", "lon_bin":"lon"})
    )
dsxr_effort = dsxr_effort.reindex(lat=lat_grid, lon=lon_grid)

dsxr_effort = dsxr_effort.assign_coords(
    lat=dsxr_effort["lat"].astype(str).astype(float),
    lon=dsxr_effort["lon"].astype(str).astype(float)
    )
valid = mask_ds["mask"]>0
dsxr_effort=dsxr_effort.where(valid)


In [13]:

#get the fishing seasons
# Keep only 2009–2025 and the two species
df_season = df1[
    df1["FechaCaptura"].dt.year.between(2024, 2025)
    & df1["EspecieFAO"].isin(["HKP", "SQA"])
].copy()

# Add month number/name
df_season["month_num"] = df_season["FechaCaptura"].dt.month
df_season["month"] = df_season["FechaCaptura"].dt.month_name()

# Total catch per species per month, pooling all years together
monthly = (
    df_season
    .groupby(["EspecieFAO", "month_num", "month"], as_index=False)["Peso"]
    .sum()
    .rename(columns={"Peso": "monthly_catch"})
)

# Percentage of each species' total catch that happened in each month
monthly["percent_of_species_total"] = (
    monthly["monthly_catch"]
    / monthly.groupby("EspecieFAO")["monthly_catch"].transform("sum")
    * 100
)

monthly = monthly.sort_values(["EspecieFAO", "month_num"])

monthly
#de aquí extraemos que 
# para HKP: de marzo a octubre mayor que el 90%
# para SQA: de diciembre a abril: mayor del 90%

,EspecieFAO,month_num,month,monthly_catch,percent_of_species_total
0,HKP,1,January,1846659.52,1.556490
1,HKP,2,February,5934614.59,5.002096
2,HKP,3,March,10326742.13,8.704079
3,HKP,4,April,17276390.75,14.561715
4,HKP,5,May,17408330.19,14.672922
5,HKP,6,June,17181558.30,14.481784
6,HKP,7,July,14527190.41,12.244502
7,HKP,8,August,14177299.33,11.949590
8,HKP,9,September,13441744.86,11.329615
9,HKP,10,October,6148351.78,5.182248


In [14]:
iters = 10
sp = "HKP"


list_of_improvements=[]
for i in range(iters):

    pred_ds = xr.open_dataset(f"./predicted/predicted_{sp}_{i +1}.nc")

    # Factor for all vessels, for normalization. Represents the importance that month for the fleet
    factor_buque = dsxr_effort["Horas"].sum(dim=("lat", "lon")) / dsxr_effort["time"].dt.days_in_month * 24
    mean_factor_buque = factor_buque.mean(dim="NombreBuque", skipna=True)

    dsxr_effort["Horas_ratio"] = (dsxr_effort["Horas"] / dsxr_effort["Horas"].sum(dim=("lat", "lon")))

    real_ds_sp = real_ds.sel(FAOspp=sp)

    dsxr_effort, pred_ds, real_ds_sp= xr.align(dsxr_effort, pred_ds,real_ds_sp, join="inner")

    # Transformation from log to normal scale
    pred_ds["pred_normal"] = np.exp(pred_ds["pred"])
    pred_ds["target_normal"] = np.exp(pred_ds["target"])


    ###seasons
    hkp_months = [4, 5, 6, 7, 8, 9]
    sqa_months = [1, 2, 3, 12]
    if sp == "HKP":
        sp_months = hkp_months
    elif sp == "SQA":
        sp_months = sqa_months

    season_sp = pred_ds["time"].dt.month.isin(sp_months)

    pred_ds_sp = pred_ds.where(season_sp, drop=True)
    dsxr_effort_sp = dsxr_effort.where(season_sp, drop=True)
    mean_factor_buque_sp = mean_factor_buque.where(season_sp, drop=True)
    factor_buque_sp = factor_buque.where(season_sp, drop=True)
    real_ds_sp = real_ds_sp.where(season_sp, drop=True)

    mean_factor_buque_sp = mean_factor_buque_sp / mean_factor_buque_sp.sum("time")
    factor_buque_sp = factor_buque_sp / factor_buque_sp.sum("time")


    target = real_ds_sp["CPUE"]

    #real fishing vessels
    weighted_sp = (dsxr_effort_sp["Horas_ratio"]*target
                    # *mean_factor_buque_sp
                )
    score_time = weighted_sp.sum(dim=("lat", "lon"), skipna=True)

    score_total = score_time.sum(dim="time", skipna=True)

    score_df = score_total.to_dataframe(name="score").reset_index()



    #####imaginary fishing vessel
    pred = pred_ds_sp["pred_normal"]

    # Stack lat/lon into one spatial dimension
    stacked = pred.stack(cell=("lat", "lon"))  # dims: time, cell

    # Convert to numpy for selecting exact top  per timestep
    values = stacked.values

    # Create empty mask
    top_values = np.zeros_like(values, dtype=np.float32)

    # Ignore NaNs when finding top 
    values_no_nan = np.where(np.isnan(values), -np.inf, values)

    # Indices of top cells for each timestep
    top_idx = np.argpartition(values_no_nan, -4, axis=1)[:, -4:]

    # Assign 0.2 to those top  cells
    rows = np.arange(values.shape[0])[:, None]
    top_values[rows, top_idx] = 0.25

    # Convert back to xarray
    top_ratio = xr.DataArray(
        top_values,
        coords=stacked.coords,
        dims=stacked.dims,
        name="top_ratio"
    ).unstack("cell")

    # Make sure dimension order is the same as pred
    top_ratio = top_ratio.transpose("time", "lat", "lon")

    # Add to pred_ds
    pred_ds_sp["top_ratio"] = top_ratio

    #imaginary score
    imaginary_weighted = (pred_ds_sp["top_ratio"]*target
                        # *mean_factor_buque_sp
                        ).sum(dim=("lat", "lon"))
    imaginary_score_total = imaginary_weighted.sum(dim="time")

    imaginary_score = imaginary_score_total.values
    vessels_active=score_df[score_df["score"]>1]
    print("imaginary:", imaginary_score)
    print("real:", vessels_active["score"].mean())
    print("imaginary ranking:", len(score_df[score_df["score"]>imaginary_score])+1,"/",len(vessels_active)+1)
    improvement_rate = ((imaginary_score - vessels_active["score"].mean())/vessels_active["score"].mean())*100
    list_of_improvements.append(improvement_rate)

array_improv = np.array(list_of_improvements)
print("--------------------------------------------------------------------------")
print(f"Improved percentage:{array_improv.mean():.2F}±{array_improv.std():.2F}")

imaginary: 1572.6107537205712
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1696.5824222548868
real: 1407.8853587669958
imaginary ranking: 10 / 25
imaginary: 1549.6404413834348
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1594.6722909652751
real: 1407.8853587669958
imaginary ranking: 11 / 25
imaginary: 1675.324273255394
real: 1407.8853587669958
imaginary ranking: 10 / 25
imaginary: 1490.047934628243
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1518.6619508618303
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1420.109625717338
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1543.410818066562
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1678.5249047542675
real: 1407.8853587669958
imaginary ranking: 10 / 25
--------------------------------------------------------------------------
Improved percentage:11.80±6.03


In [15]:

list_of_improvements=[]
for i in range(iters):

    pred_ds = xr.open_dataset(f"./predicted/predicted_RF_{sp}_{i +1}.nc")

    # Factor for all vessels, for normalization. Represents the importance that month for the fleet
    factor_buque = dsxr_effort["Horas"].sum(dim=("lat", "lon")) / dsxr_effort["time"].dt.days_in_month * 24
    mean_factor_buque = factor_buque.mean(dim="NombreBuque", skipna=True)

    dsxr_effort["Horas_ratio"] = (dsxr_effort["Horas"] / dsxr_effort["Horas"].sum(dim=("lat", "lon")))

    real_ds_sp = real_ds.sel(FAOspp=sp)

    dsxr_effort, pred_ds, real_ds_sp= xr.align(dsxr_effort, pred_ds,real_ds_sp, join="inner")

    # Transformation from log to normal scale
    pred_ds["pred_normal"] = np.exp(pred_ds["pred"])
    pred_ds["target_normal"] = np.exp(pred_ds["target"])


    ###seasons
    hkp_months = [4, 5, 6, 7, 8, 9]
    sqa_months = [1, 2, 3, 12]
    if sp == "HKP":
        sp_months = hkp_months
    elif sp == "SQA":
        sp_months = sqa_months

    season_sp = pred_ds["time"].dt.month.isin(sp_months)

    pred_ds_sp = pred_ds.where(season_sp, drop=True)
    dsxr_effort_sp = dsxr_effort.where(season_sp, drop=True)
    mean_factor_buque_sp = mean_factor_buque.where(season_sp, drop=True)
    factor_buque_sp = factor_buque.where(season_sp, drop=True)
    real_ds_sp = real_ds_sp.where(season_sp, drop=True)

    mean_factor_buque_sp = mean_factor_buque_sp / mean_factor_buque_sp.sum("time")
    factor_buque_sp = factor_buque_sp / factor_buque_sp.sum("time")


    target = real_ds_sp["CPUE"]

    #real fishing vessels
    weighted_sp = (dsxr_effort_sp["Horas_ratio"]*target
                    # *mean_factor_buque_sp
                )
    score_time = weighted_sp.sum(dim=("lat", "lon"), skipna=True)

    score_total = score_time.sum(dim="time", skipna=True)

    score_df = score_total.to_dataframe(name="score").reset_index()



    #####imaginary fishing vessel
    pred = pred_ds_sp["pred_normal"]

    # Stack lat/lon into one spatial dimension
    stacked = pred.stack(cell=("lat", "lon"))  # dims: time, cell

    # Convert to numpy for selecting exact top  per timestep
    values = stacked.values

    # Create empty mask
    top_values = np.zeros_like(values, dtype=np.float32)

    # Ignore NaNs when finding top 
    values_no_nan = np.where(np.isnan(values), -np.inf, values)

    # Indices of top cells for each timestep
    top_idx = np.argpartition(values_no_nan, -4, axis=1)[:, -4:]

    # Assign 0.2 to those top  cells
    rows = np.arange(values.shape[0])[:, None]
    top_values[rows, top_idx] = 0.25

    # Convert back to xarray
    top_ratio = xr.DataArray(
        top_values,
        coords=stacked.coords,
        dims=stacked.dims,
        name="top_ratio"
    ).unstack("cell")

    # Make sure dimension order is the same as pred
    top_ratio = top_ratio.transpose("time", "lat", "lon")

    # Add to pred_ds
    pred_ds_sp["top_ratio"] = top_ratio

    #imaginary score
    imaginary_weighted = (pred_ds_sp["top_ratio"]*target
                        # *mean_factor_buque_sp
                        ).sum(dim=("lat", "lon"))
    imaginary_score_total = imaginary_weighted.sum(dim="time")

    imaginary_score = imaginary_score_total.values
    vessels_active=score_df[score_df["score"]>1]
    print("imaginary:", imaginary_score)
    print("real:", vessels_active["score"].mean())
    print("imaginary ranking:", len(score_df[score_df["score"]>imaginary_score])+1,"/",len(vessels_active)+1)
    improvement_rate = ((imaginary_score - vessels_active["score"].mean())/vessels_active["score"].mean())*100
    list_of_improvements.append(improvement_rate)

array_improv = np.array(list_of_improvements)
print("--------------------------------------------------------------------------")
print(f"Improved percentage:{array_improv.mean():.2F}±{array_improv.std():.2F}")

imaginary: 1572.5248708509675
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1490.4360383010412
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1495.1609924244046
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1489.856736997097
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1507.403597312556
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1565.7997017974074
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1551.5434022585757
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1583.912290705183
real: 1407.8853587669958
imaginary ranking: 12 / 25
imaginary: 1498.05801132897
real: 1407.8853587669958
imaginary ranking: 13 / 25
imaginary: 1531.7398658157877
real: 1407.8853587669958
imaginary ranking: 13 / 25
--------------------------------------------------------------------------
Improved percentage:8.58±2.50


In [16]:
pred_ds = xr.open_dataset(f"./predicted/predicted_BL_{sp}.nc")

# Factor for all vessels, for normalization. Represents the importance that month for the fleet
factor_buque = dsxr_effort["Horas"].sum(dim=("lat", "lon")) / dsxr_effort["time"].dt.days_in_month * 24
mean_factor_buque = factor_buque.mean(dim="NombreBuque", skipna=True)

dsxr_effort["Horas_ratio"] = (dsxr_effort["Horas"] / dsxr_effort["Horas"].sum(dim=("lat", "lon")))

real_ds_sp = real_ds.sel(FAOspp=sp)

dsxr_effort, pred_ds, real_ds_sp= xr.align(dsxr_effort, pred_ds,real_ds_sp, join="inner")

# Transformation from log to normal scale
pred_ds["pred_normal"] = np.exp(pred_ds["pred"])
pred_ds["target_normal"] = np.exp(pred_ds["target"])


###seasons
hkp_months = [4, 5, 6, 7, 8, 9]
sqa_months = [1, 2, 3, 12]
if sp == "HKP":
    sp_months = hkp_months
elif sp == "SQA":
    sp_months = sqa_months

season_sp = pred_ds["time"].dt.month.isin(sp_months)

pred_ds_sp = pred_ds.where(season_sp, drop=True)
dsxr_effort_sp = dsxr_effort.where(season_sp, drop=True)
mean_factor_buque_sp = mean_factor_buque.where(season_sp, drop=True)
factor_buque_sp = factor_buque.where(season_sp, drop=True)
real_ds_sp = real_ds_sp.where(season_sp, drop=True)

mean_factor_buque_sp = mean_factor_buque_sp / mean_factor_buque_sp.sum("time")
factor_buque_sp = factor_buque_sp / factor_buque_sp.sum("time")


target = real_ds_sp["CPUE"]

#real fishing vessels
weighted_sp = (dsxr_effort_sp["Horas_ratio"]*target
                # *mean_factor_buque_sp
            )
score_time = weighted_sp.sum(dim=("lat", "lon"), skipna=True)

score_total = score_time.sum(dim="time", skipna=True)

score_df = score_total.to_dataframe(name="score").reset_index()



#####imaginary fishing vessel
pred = pred_ds_sp["pred_normal"]

# Stack lat/lon into one spatial dimension
stacked = pred.stack(cell=("lat", "lon"))  # dims: time, cell

# Convert to numpy for selecting exact top  per timestep
values = stacked.values

# Create empty mask
top_values = np.zeros_like(values, dtype=np.float32)

# Ignore NaNs when finding top 
values_no_nan = np.where(np.isnan(values), -np.inf, values)

# Indices of top cells for each timestep
top_idx = np.argpartition(values_no_nan, -4, axis=1)[:, -4:]

# Assign 0.2 to those top  cells
rows = np.arange(values.shape[0])[:, None]
top_values[rows, top_idx] = 0.25

# Convert back to xarray
top_ratio = xr.DataArray(
    top_values,
    coords=stacked.coords,
    dims=stacked.dims,
    name="top_ratio"
).unstack("cell")

# Make sure dimension order is the same as pred
top_ratio = top_ratio.transpose("time", "lat", "lon")

# Add to pred_ds
pred_ds_sp["top_ratio"] = top_ratio

#imaginary score
imaginary_weighted = (pred_ds_sp["top_ratio"]*target
                    # *mean_factor_buque_sp
                    ).sum(dim=("lat", "lon"))
imaginary_score_total = imaginary_weighted.sum(dim="time")

imaginary_score = imaginary_score_total.values
vessels_active=score_df[score_df["score"]>1]
print("imaginary:", imaginary_score)
print("real:", vessels_active["score"].mean())
print("imaginary ranking:", len(score_df[score_df["score"]>imaginary_score])+1,"/",len(vessels_active)+1)
improvement_rate = ((imaginary_score - vessels_active["score"].mean())/vessels_active["score"].mean())*100
print(improvement_rate)

imaginary: 1895.6879916594212
real: 1407.8853587669958
imaginary ranking: 5 / 25
34.64789443649271
